# Project 7 — Email Spam Detection

## Goal
Classify each email as **SPAM** (unwanted) or **HAM** (legitimate).

## Dataset
`email_data/spam_ham_dataset.csv` with columns:
- `label` — *ham* or *spam*
- `text`  — the raw email body (often starts with `Subject: …`)

## Pipeline
1. Load the CSV
2. Clean each email
3. TF-IDF (unigrams + bigrams)
4. Train Naive Bayes & Logistic Regression
5. Compare, predict, interpret

## Step 1 — Imports

In [ ]:
import os, re, string
import numpy as np, pandas as pd
import nltk, spacy

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

for pkg in ['stopwords', 'punkt', 'punkt_tab', 'wordnet']:
    nltk.download(pkg, quiet=True)
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nlp = spacy.load('en_core_web_sm', disable=['parser', 'ner'])

## Step 2 — Load the data

In [ ]:
DATA_PATH = os.path.join('email_data', 'spam_ham_dataset.csv')
df = pd.read_csv(DATA_PATH)
print(f'Total emails: {len(df):,}')
print('Columns     :', df.columns.tolist())
df.head(3)

In [ ]:
print('Class distribution:')
print(df['label'].value_counts())
print('\nExample SPAM:\n', df[df.label=='spam'].text.iloc[0][:200])

## Step 3 — Clean each email

Real emails contain `Subject:` headers, URLs, email addresses, HTML tags, numbers, and lots of punctuation.

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_email_nltk(text):
    text = str(text).lower()
    text = re.sub(r'^subject\s*:\s*', ' ', text)
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = re.sub(r'[^a-z\s]', ' ', text)
    tokens = [lemmatizer.lemmatize(t) for t in text.split()
              if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

def clean_email_spacy(text):
    text = re.sub(r'http\S+|\S+@\S+|<[^>]+>|\d+', ' ', str(text).lower())
    doc = nlp(text)
    return ' '.join(tok.lemma_ for tok in doc
                    if tok.is_alpha and not tok.is_stop and len(tok.text) > 2)

sample = df.text.iloc[0]
print('Original:', sample[:120])
print('NLTK    :', clean_email_nltk(sample)[:120])
print('spaCy   :', clean_email_spacy(sample)[:120])

In [ ]:
df['clean'] = df['text'].apply(clean_email_nltk)
df = df[df['clean'].str.len() > 0].reset_index(drop=True)
print(f'Emails after cleaning: {len(df):,}')

## Step 4 — Train/Test split

In [ ]:
y = (df['label'] == 'spam').astype(int).values
X_text = df['clean'].values

X_train, X_test, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y)
print(f'Train: {len(X_train):,}    Test: {len(X_test):,}')

## Step 5 — TF-IDF
Bigrams catch common spam phrases like *click here*, *free money*, *act now*.

In [ ]:
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), min_df=2)
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec  = vectorizer.transform(X_test)
print('Train matrix:', X_train_vec.shape)

## Step 6 — Train two classifiers

In [ ]:
nb = MultinomialNB().fit(X_train_vec, y_train)
lr = LogisticRegression(max_iter=1000, n_jobs=-1).fit(X_train_vec, y_train)

print(f'Naive Bayes        accuracy = {accuracy_score(y_test, nb.predict(X_test_vec)):.4f}')
print(f'Logistic Regression accuracy = {accuracy_score(y_test, lr.predict(X_test_vec)):.4f}')

print('\nLogistic Regression report:')
print(classification_report(y_test, lr.predict(X_test_vec), target_names=['Ham', 'Spam']))

## Step 7 — Confusion matrix

```
                Predicted Ham   Predicted Spam
Actual Ham           TN              FP
Actual Spam          FN              TP
```

- **FP (false positive)** — a real email marked as spam → BAD (user misses important mail)
- **FN (false negative)** — spam slipped into the inbox → ANNOYING but recoverable

In [ ]:
best = lr if accuracy_score(y_test, lr.predict(X_test_vec)) >= accuracy_score(y_test, nb.predict(X_test_vec)) else nb
print(f'Best model: {type(best).__name__}')
print(confusion_matrix(y_test, best.predict(X_test_vec)))

## Step 8 — Predict on new emails

In [ ]:
new_emails = [
    'Congratulations! You have WON a $1000 Walmart Gift Card. Click here to claim',
    'Hi John, please find attached the project report for our 9 AM meeting',
    'URGENT: Your account has been compromised. Verify your password immediately',
    'Reminder: Lunch with Sarah tomorrow at 1pm at the Italian place',
    'Get cheap meds online! 100% discreet shipping, no prescription needed',
]
X_new = vectorizer.transform([clean_email_nltk(e) for e in new_emails])
for e, p, prob in zip(new_emails, best.predict(X_new), best.predict_proba(X_new)):
    label = 'SPAM' if p == 1 else 'HAM'
    print(f'  [{label} {prob[p]:.0%}]  {e[:65]}…')

## Step 9 — Top spam-indicator words

In [ ]:
if isinstance(best, LogisticRegression):
    feats = np.array(vectorizer.get_feature_names_out())
    coefs = best.coef_[0]
    print('TOP SPAM words/phrases:')
    for i in np.argsort(coefs)[-15:][::-1]:
        print(f'  {feats[i]:<25} {coefs[i]:+.3f}')
    print('\nTOP HAM words/phrases:')
    for i in np.argsort(coefs)[:15]:
        print(f'  {feats[i]:<25} {coefs[i]:+.3f}')

## Summary

You built a spam classifier with ~98% accuracy.

**Ideas to extend this:**
- Add character-level n-grams to catch obfuscated words (`v1@gra`)
- Add metadata features (count of `$`, all-caps ratio, link count)
- Replace TF-IDF with a fine-tuned BERT for production